# 01 - Dataset & preprocessing (Member 1)
Goal: verified input/target pairs, documented split, untouched official test set. **Stop after verifying pairs.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, yaml
REPO = "/content/nighttime-flare-removal"
if not os.path.exists(REPO):
    !git clone https://github.com/USERNAME/nighttime-flare-removal.git {REPO}   # <-- replace USERNAME
%cd {REPO}
!git pull
!pip -q install pyyaml scikit-image tqdm
sys.path.insert(0, REPO)
cfg = yaml.safe_load(open("configs/default.yaml"))
import torch
print("CUDA:", torch.cuda.is_available())

In [ ]:
PROJECT = cfg["project"]["drive_root"]
for f in ["data/archives","checkpoints/baseline","checkpoints/improved","results/metrics","results/figures","logs","splits"]:
    os.makedirs(os.path.join(PROJECT, f), exist_ok=True)
print(os.listdir(PROJECT + "/data/archives"))   # note the exact archive filenames

## Clone the official repo and extract archives
Download Flare7K++ and the background archive from the official README (Data Download section) into `data/archives/`.

In [ ]:
%cd /content
!git clone https://github.com/ykdai/Flare7K.git
!mkdir -p /content/flare_data
# Replace with the EXACT filenames printed above:
# !unzip -q "{PROJECT}/data/archives/FLARE_ARCHIVE.zip" -d /content/flare_data/
# !unzip -q "{PROJECT}/data/archives/BACKGROUND_ARCHIVE.zip" -d /content/flare_data/
!find /content/flare_data -maxdepth 4 -type d | head -100

## Official generator / loader
Open `/content/Flare7K/generate_flare.ipynb`, read its path cells, change only paths, run a SMALL sample first.
Write pairs to `/content/flare_pairs/input/` and `/content/flare_pairs/target/` with identical filenames (`<bg>__<flare>.png`). See `docs/DATA_NOTES.md`.

In [ ]:
%cd {REPO}
from src.data.dataset import build_loaders, PairedFolderDataset, verify_pairs
import os
d = cfg["data"]
ds = PairedFolderDataset(os.path.join(d["pairs_root"], "input"), os.path.join(d["pairs_root"], "target"),
                         crop_size=d["crop_size"], train=False)
print(len(ds), "pairs")
verify_pairs(ds, n=5, save_path=PROJECT + "/results/figures/verified_pairs.png")

In [ ]:
train_loader, val_loader = build_loaders(cfg)
b = next(iter(train_loader))
print(b["input"].shape, b["target"].shape, b["input"].min().item(), b["input"].max().item())
print("train batches:", len(train_loader), "val batches:", len(val_loader))
# split JSON is saved to Drive -> commit its summary (counts) to docs/DATA_NOTES.md, NOT the data